In [3]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

In [41]:
from functools import partial
from pathlib import Path

import coremltools as ct
import coremltools.optimize as cto
import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm

In [5]:
def make_net(n_classes):
    base_arch = keras.applications.MobileNetV3Large
    base_model = base_arch(
        input_shape=[224, 224, 3],
        weights="imagenet",
        include_top=False
    )
    for layer in base_model.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True
        
    model = keras.Sequential([
        base_model,
        keras.layers.GlobalAveragePooling2D(keepdims=True),
        keras.layers.Dropout(0.2),
        keras.layers.Conv2D(64, [1, 1], name="svd_u"),
        keras.layers.Conv2D(n_classes, [1, 1], name="logits"),
        keras.layers.Reshape([n_classes], name="reshaped_logits"),
    ])

    return model

In [15]:
tqdm.pandas()

# make dataset

In [6]:
export_dir = Path("/data-ssd/alex/datasets/vision-export-20221020154449-Galliformes-573/")

In [7]:
tax = pd.read_csv(export_dir / "taxonomy.csv")
leaf_tax = tax[~tax.leaf_class_id.isna()]
num_classes = len(leaf_tax)
print(f"{num_classes} classes")

122 classes


In [8]:

train_augmented = pd.read_csv(export_dir / "train_data_augmented.csv")

In [9]:
!ls {export_dir}

iconic_taxa.csv      test_data.csv	       train_download_errs.csv
spatial_data.csv     test_download_errs.csv    val_cleaned.csv
taxonomy.csv	     train_cleaned.csv	       val_data.csv
taxonomy_visual.txt  train_data_augmented.csv  val_download_errs.csv
test_cleaned.csv     train_data.csv


In [10]:
train_augmented

,photo_id,extension,taxon_id,leaf_class_id
0,21403,jpg,2038,26
1,21406,jpg,1406,23
2,21407,jpg,1406,23
3,21774,jpg,981,117
4,22200,jpg,1409,85
...,...,...,...,...
420136,424021455,jpg,906,116
420137,424035822,jpg,558443,47
420138,424035833,jpg,558443,47
420139,424035825,jpg,558443,47


In [11]:
import hashlib
import os

def path_for_photo_id(base_dir, photo_id):
    msg = str(photo_id).encode('utf-8')
    m = hashlib.sha256()
    m.update(msg)
    hashed = m.hexdigest()
    filename = '{}.jpg'.format(photo_id)
    photo_paths = [base_dir, hashed[0:2], hashed[2:4], filename]
    return os.path.join(*photo_paths)

In [18]:
train_augmented["filename"] = train_augmented.progress_apply(lambda x: path_for_photo_id("/data-ssd2/alex/images", x.photo_id), axis=1)

100%|████████████████████████████████████████████████████████████████████████████| 420141/420141 [00:02<00:00, 144553.85it/s]


In [19]:
train_cleaned = pd.read_csv(
    export_dir / "train_cleaned.csv",
    usecols=["filename", "leaf_class_id"],
)

In [20]:
train_augmented["file_exists"] = train_augmented.filename.progress_apply(lambda x: os.path.exists(x))

100%|████████████████████████████████████████████████████████████████████████████| 420141/420141 [00:01<00:00, 329102.07it/s]


In [21]:
train_augmented = train_augmented[train_augmented.file_exists]

In [22]:
train_cleaned = train_cleaned[["filename", "leaf_class_id"]]
train_augmented = train_augmented[["filename", "leaf_class_id"]]
train_df = pd.concat([train_cleaned, train_augmented])
train_df.drop_duplicates(inplace=True, subset=["filename"], keep=False)

In [23]:
train_df.leaf_class_id.value_counts()

leaf_class_id
116    134142
117     40895
85      38140
120     29773
33      21359
        ...  
70        138
101       135
106       100
76         91
71         90
Name: count, Length: 122, dtype: int64

In [24]:
max_sample_size = 1_000

train_df = train_df.sample(frac=1)
train_df_sample = train_df.groupby("leaf_class_id").head(max_sample_size)

In [25]:
train_df_sample.leaf_class_id.value_counts()

leaf_class_id
77     1000
40     1000
80     1000
60     1000
38     1000
       ... 
70      138
101     135
106     100
76       91
71       90
Name: count, Length: 122, dtype: int64

In [26]:
train_cleaned.leaf_class_id.value_counts()

leaf_class_id
118    1000
48     1000
23     1000
104    1000
82     1000
       ... 
50       63
17       58
29       53
101      50
71       50
Name: count, Length: 122, dtype: int64

# baseline

In [27]:
BATCH_SIZE=128
IMAGE_SIZE=[224, 224]

In [28]:
def _flip(x, y):
    x = tf.image.random_flip_left_right(x)
    return x, y

def _random_crop(x, y):
    bbox = tf.constant([0.0, 0.0, 1.0, 1.0], dtype=tf.float32, shape=[1, 1, 4])
    begin, size, bbox_for_draw = tf.image.sample_distorted_bounding_box(
        tf.shape(x),
        bounding_boxes=bbox,
        area_range=(0.08, 1.0),
        aspect_ratio_range=(0.75, 1.33),
        max_attempts=100,
        min_object_covered=0.1,
    )
    x = tf.slice(x, begin, size)
    return x, y

def _color(x, y):
    x = tf.image.random_hue(x, 0.08)
    x = tf.image.random_saturation(x, 0.6, 1.6)
    x = tf.image.random_brightness(x, 0.05)
    x = tf.image.random_contrast(x, 0.7, 1.3)
    return x, y

# mobilenets v3 expects images in range 0,255
def _decode_img(img):
    img = tf.image.decode_jpeg(img, channels=3)
    return img

def _process(filepath, label, num_classes):
    img = tf.io.read_file(filepath)
    img = _decode_img(img)
    label = tf.one_hot(label, num_classes)
    return img, label

def make_dataset(dataframe, BATCH_SIZE, IMAGE_SIZE, repeat_forever=True, augment=False):
    num_examples = len(dataframe)
    ds = tf.data.Dataset.from_tensor_slices(
        (dataframe["filename"], dataframe["leaf_class_id"])
    )
    ds = ds.shuffle(buffer_size=10_000, reshuffle_each_iteration=True)
    process_partial = partial(_process, num_classes=num_classes)
    ds = ds.map(process_partial, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        ds = ds.map(lambda x, y: _flip(x, y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda x, y: _random_crop(x, y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda x, y: (x / 255.0, y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda x, y: _color(x, y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda x, y: (tf.clip_by_value(x, 0, 1), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda x, y: (x * 255.0, y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda x, y: (tf.cast(x, tf.uint8), y), num_parallel_calls=tf.data.AUTOTUNE)

        
        ds = ds.map(lambda x, y: (tf.image.resize(x, IMAGE_SIZE), y), num_parallel_calls=tf.data.AUTOTUNE)

    else:
        # no augments
        ds = ds.map(lambda x, y: (tf.image.central_crop(x, 0.875), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda x, y: (tf.image.resize(x, IMAGE_SIZE), y), num_parallel_calls=tf.data.AUTOTUNE)
        
    ds = ds.batch(BATCH_SIZE)
    if repeat_forever:
        ds = ds.repeat()
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    
    return ds, num_examples

In [29]:
train_cleaned = pd.read_csv(export_dir / "train_cleaned.csv")
train_cleaned = train_cleaned.sample(frac=1)
val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
val_cleaned = val_cleaned.sample(frac=1)

(train_ds, num_train_examples) = make_dataset(train_cleaned, BATCH_SIZE, IMAGE_SIZE)
(val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE)

In [30]:
# mixed precision training
policy = keras.mixed_precision.Policy("mixed_float16")
keras.mixed_precision.set_global_policy(policy)

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA RTX 6000 Ada Generation, compute capability 8.9


In [31]:
model = make_net(num_classes)

In [32]:
model.compile(
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["accuracy"],
)

In [33]:
STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))

In [34]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    validation_steps=VAL_STEPS_PER_EPOCH,
    epochs=2,
    steps_per_epoch=STEPS_PER_EPOCH
)

Epoch 1/2


I0000 00:00:1725891568.286247 1927596 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


517/517 [==============================] - 42s 61ms/step - loss: 3.1957 - accuracy: 0.2800 - val_loss: 2.5427 - val_accuracy: 0.4242
Epoch 2/2
517/517 [==============================] - 30s 58ms/step - loss: 1.8870 - accuracy: 0.5240 - val_loss: 1.9849 - val_accuracy: 0.5183


In [35]:
model.save("galliformes_models/baseline.h5")

/home/alex/inatVisionTrainingMain/inatVisionTraining/venv/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [39]:
image_input = ct.ImageType(
    shape=(1, 224, 224, 3,),
)
coreml_model = ct.convert(
    model,
    inputs=[image_input],
    minimum_deployment_target=ct.target.iOS13,
)

Translating MIL ==> NeuralNetwork Ops: 100%|█████████████████████████████████████████| 1396/1396 [00:00<00:00, 7459.87 ops/s]


In [40]:
coreml_model.save("galliformes_models/baseline.mlmodel")

In [43]:
!ls -lah galliformes_models/

total 48M
drwxr-xr-x  2 alex alex 4.0K Sep  9 07:27 .
drwxr-xr-x 12 alex alex 4.0K Sep  9 07:27 ..
-rw-rw-r--  1 alex alex  36M Sep  9 07:20 baseline.h5
-rw-rw-r--  1 alex alex  12M Sep  9 07:27 baseline.mlmodel


In [44]:
!pwd

/home/alex/inatVisionTrainingMain/inatVisionTraining


In [45]:
op_config = cto.coreml.OpLinearQuantizerConfig(
    mode="linear_symmetric", weight_threshold=512,
)
config = cto.coreml.OptimizationConfig(global_config=op_config)

In [49]:
coreml_model_compressed = ct.models.neural_network.quantization_utils.quantize_weights(
    coreml_model,
    nbits=8,
)

Quantizing using linear quantization
Optimizing Neural Network before Quantization:
Finished optimizing network. Quantizing neural network..
Quantizing layer sequential/MobilenetV3large/Conv/Conv2Dx of type convolution
Quantizing layer sequential/MobilenetV3large/Conv/BatchNorm/FusedBatchNormV3_nchw of type batchnorm
Quantizing layer sequential/MobilenetV3large/expanded_conv/depthwise/depthwisex of type convolution
Quantizing layer sequential/MobilenetV3large/expanded_conv/depthwise/BatchNorm/FusedBatchNormV3_nchw of type batchnorm
Quantizing layer sequential/MobilenetV3large/expanded_conv/project/Conv2Dx of type convolution
Quantizing layer sequential/MobilenetV3large/expanded_conv/project/BatchNorm/FusedBatchNormV3_nchw of type batchnorm
Quantizing layer sequential/MobilenetV3large/expanded_conv_1/expand/Conv2Dx of type convolution
Quantizing layer sequential/MobilenetV3large/expanded_conv_1/expand/BatchNorm/FusedBatchNormV3_nchw of type batchnorm
Quantizing layer sequential/Mobilene

Quantizing layer conv_15 of type convolution
Quantizing layer sequential/MobilenetV3large/expanded_conv_13/project/Conv2Dx of type convolution
Quantizing layer sequential/MobilenetV3large/expanded_conv_13/project/BatchNorm/FusedBatchNormV3_nchw of type batchnorm
Quantizing layer sequential/MobilenetV3large/expanded_conv_14/expand/Conv2Dx of type convolution
Quantizing layer sequential/MobilenetV3large/expanded_conv_14/expand/BatchNorm/FusedBatchNormV3_nchw of type batchnorm
Quantizing layer sequential/MobilenetV3large/expanded_conv_14/depthwise/depthwisex of type convolution
Quantizing layer sequential/MobilenetV3large/expanded_conv_14/depthwise/BatchNorm/FusedBatchNormV3_nchw of type batchnorm
Quantizing layer sequential/MobilenetV3large/expanded_conv_14/squeeze_excite/Conv/BiasAdd of type convolution
Quantizing layer conv_16 of type convolution
Quantizing layer sequential/MobilenetV3large/expanded_conv_14/project/Conv2Dx of type convolution
Quantizing layer sequential/MobilenetV3larg

In [50]:
coreml_model_compressed.save("galliformes_models/baseline_8bit.mlmodel")

In [51]:
!ls -lah galliformes_models/

total 51M
drwxr-xr-x  2 alex alex 4.0K Sep  9 07:37 .
drwxr-xr-x 12 alex alex 4.0K Sep  9 07:36 ..
-rw-rw-r--  1 alex alex 3.2M Sep  9 07:37 baseline_8bit.mlmodel
-rw-rw-r--  1 alex alex  36M Sep  9 07:20 baseline.h5
-rw-rw-r--  1 alex alex  12M Sep  9 07:27 baseline.mlmodel


# ok let's do with a custom training loop for comparison

In [ ]:
train_cleaned = pd.read_csv(export_dir / "train_cleaned.csv")
train_cleaned = train_cleaned.sample(frac=1)
val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
val_cleaned = val_cleaned.sample(frac=1)

(train_ds, num_train_examples) = make_dataset(train_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)
(val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)

STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))

In [ ]:
loss = keras.losses.CategoricalCrossentropy(from_logits=True)
optimizer = keras.optimizers.Adam(learning_rate=1e-4)
accuracy = keras.metrics.Accuracy()
model = make_net(num_classes)

for epoch in range(2):
    print(f"epoch {epoch+1}")
    losses = []
    with tqdm(total=STEPS_PER_EPOCH as pbar:
        for step, (x, y) in enumerate(train_ds):
            with tf.GradientTape() as tape:
                yhat = model(x, training=True)
                batch_loss = loss(y, yhat)
                grads = tape.gradient(batch_loss, model.trainable_weights)
                optimizer.apply_gradients(zip(grads, model.trainable_weights))
            pbar.update()
            losses.append(batch_loss)
        print(f"epoch loss {np.mean(losses)}")


In [ ]:
model.compile(
    loss=loss,
    metrics=["accuracy"]
)

In [ ]:
model.evaluate(val_ds)

# ok let's try one dataset per epoch (stupid but w/e)

In [ ]:
baseline_model = make_net(num_classes)
baseline_model.compile(
    loss=loss,
    metrics=["accuracy"],
    optimizer=keras.optimizers.Adam(1e-4),
)

history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    print(f"epoch {epoch}")
    train_cleaned = pd.read_csv(export_dir / "train_cleaned.csv")
    train_cleaned = train_cleaned.sample(frac=1)
    val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
    val_cleaned = val_cleaned.sample(frac=1)

    (train_ds, num_train_examples) = make_dataset(train_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)
    (val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)

    STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
    VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))
    
    epoch_hist = baseline_model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_data=val_ds,
        validation_steps=VAL_STEPS_PER_EPOCH,
        epochs=epoch+1,
        initial_epoch=epoch,
    )
    
    history["loss"].append(epoch_hist.history["loss"][-1])
    history["accuracy"].append(epoch_hist.history["accuracy"][-1])
    history["val_loss"].append(epoch_hist.history["val_loss"][-1])
    history["val_accuracy"].append(epoch_hist.history["val_accuracy"][-1])



In [ ]:

baseline_hist = history
plt.plot(baseline_hist["val_accuracy"])


# ok now let's sample from larger dataset

In [ ]:
train_plus_aug_df = pd.concat([train_cleaned, train_augmented])
max_sample_size = 1_000

model = make_net(num_classes)
model.compile(
    loss=loss,
    metrics=["accuracy"],
    optimizer=keras.optimizers.Adam(1e-4),
)

history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

NUM_EPOCHS = 10
for epoch in range(NUM_EPOCHS):
    print(f"epoch {epoch}")
    
    train_plus_aug_df = train_plus_aug_df.sample(frac=1)
    train_df_sample = train_plus_aug_df.groupby("leaf_class_id").head(max_sample_size)
    train_df_sample = train_df_sample.sample(frac=1)
    
    val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
    val_cleaned = val_cleaned.sample(frac=1)
    
    (train_ds, num_train_examples) = make_dataset(train_df_sample, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)
    (val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)

    STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
    VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))
    
    epoch_hist = model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_data=val_ds,
        validation_steps=VAL_STEPS_PER_EPOCH,
        epochs=epoch+1,
        initial_epoch=epoch,
    )
    history["loss"].append(epoch_hist.history["loss"][-1])
    history["accuracy"].append(epoch_hist.history["accuracy"][-1])
    history["val_loss"].append(epoch_hist.history["val_loss"][-1])
    history["val_accuracy"].append(epoch_hist.history["val_accuracy"][-1])


In [ ]:
plt.plot(history["val_accuracy"])
plt.plot(baseline_hist["val_accuracy"])

# let's try with a sample size of 500

In [ ]:
train_plus_aug_df = pd.concat([train_cleaned, train_augmented])
max_sample_size = 500

model = make_net(num_classes)
model.compile(
    loss=loss,
    metrics=["accuracy"],
    optimizer=keras.optimizers.Adam(1e-4),
)

history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

NUM_EPOCHS = 20
for epoch in range(NUM_EPOCHS):
    print(f"epoch {epoch}")
    
    train_plus_aug_df = train_plus_aug_df.sample(frac=1)
    train_df_sample = train_plus_aug_df.groupby("leaf_class_id").head(max_sample_size)
    train_df_sample = train_df_sample.sample(frac=1)
    
    val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
    val_cleaned = val_cleaned.sample(frac=1)
    
    (train_ds, num_train_examples) = make_dataset(train_df_sample, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)
    (val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)

    STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
    VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))
    
    epoch_hist = model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_data=val_ds,
        validation_steps=VAL_STEPS_PER_EPOCH,
        epochs=epoch+1,
        initial_epoch=epoch,
    )
    history["loss"].append(epoch_hist.history["loss"][-1])
    history["accuracy"].append(epoch_hist.history["accuracy"][-1])
    history["val_loss"].append(epoch_hist.history["val_loss"][-1])
    history["val_accuracy"].append(epoch_hist.history["val_accuracy"][-1])


In [ ]:
plt.plot(history["val_accuracy"])
plt.plot(baseline_hist["val_accuracy"])

In [ ]:
preds = model.predict(val_ds)

# ok let's do pre class metrics with the test dataset

In [ ]:
test_cleaned = pd.read_csv(export_dir / "test_cleaned.csv")
num_test_examples = len(test_cleaned)
test_ds = tf.data.Dataset.from_tensor_slices(
    (test_cleaned["filename"], test_cleaned["leaf_class_id"])
)
process_partial = partial(_process, num_classes=num_classes)
test_ds = test_ds.map(process_partial)
test_ds = test_ds.map(lambda x, y: (tf.image.central_crop(x, 0.875), y))
test_ds = test_ds.map(lambda x, y: (tf.image.resize(x, IMAGE_SIZE), y))
test_ds = test_ds.batch(BATCH_SIZE)

In [ ]:
baseline_test_preds = baseline_model.predict(test_ds)

In [ ]:
yhats = baseline_test_preds.argmax(axis=1)
ys = test_cleaned["leaf_class_id"].values
results = pd.DataFrame({
    "y": ys,
    "yhat": yhats,
})
(results.y == results.yhat).sum() / len(results)

In [ ]:
subsets = []
for leaf_class_id in range(num_classes):
    num_train_classes = len(train_cleaned[train_cleaned.leaf_class_id==leaf_class_id])
    subset = results[(results.y == leaf_class_id)]
    subset_acc = (subset.y == subset.yhat).sum() / len(subset)
    subsets.append( (num_train_classes, subset_acc) )

(train_classes, accs) = list(zip(*subsets))
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(train_classes, accs)
baseline_b, baseline_a = np.polyfit(train_classes, accs, deg=1)
xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, baseline_a + baseline_b * xseq, color="k", lw=2.5)
plt.show()

In [ ]:
test_preds_1k = model.predict(test_ds)

In [ ]:
yhats = test_preds_1k.argmax(axis=1)
ys = test_cleaned["leaf_class_id"].values
results = pd.DataFrame({
    "y": ys,
    "yhat": yhats,
})
(results.y == results.yhat).sum() / len(results)

In [ ]:
subsets = []
for leaf_class_id in range(num_classes):
    num_train_classes = len(train_cleaned[train_cleaned.leaf_class_id==leaf_class_id])
    subset = results[(results.y == leaf_class_id)]
    subset_acc = (subset.y == subset.yhat).sum() / len(subset)
    subsets.append( (num_train_classes, subset_acc) )

(train_classes, accs) = list(zip(*subsets))
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(train_classes, accs)
b_1k, a_1k = np.polyfit(train_classes, accs, deg=1)
xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_1k + b_1k * xseq, color="k", lw=2.5)
plt.show()

In [ ]:
test_preds_500 = model.predict(test_ds)

yhats = test_preds_500.argmax(axis=1)
ys = test_cleaned["leaf_class_id"].values
results = pd.DataFrame({
    "y": ys,
    "yhat": yhats,
})
(results.y == results.yhat).sum() / len(results)

In [ ]:
subsets = []
for leaf_class_id in range(num_classes):
    num_train_classes = len(train_cleaned[train_cleaned.leaf_class_id==leaf_class_id])
    subset = results[(results.y == leaf_class_id)]
    subset_acc = (subset.y == subset.yhat).sum() / len(subset)
    subsets.append( (num_train_classes, subset_acc) )

(train_classes, accs) = list(zip(*subsets))
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(train_classes, accs)
b_500, a_500 = np.polyfit(train_classes, accs, deg=1)
xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_500 + b_500 * xseq, color="k", lw=2.5)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_500 + b_500 * xseq, lw=2.5, label="500")
ax.plot(xseq, a_1k + b_1k * xseq, lw=2.5, label="1k")
ax.plot(xseq, baseline_a + baseline_b * xseq, lw=2.5, label="baseline")


ax.set_ylim([0, 1])

ax.legend()
plt.show()

# ok let's go with augments, baseline first

In [ ]:
baseline_model = make_net(num_classes)
baseline_model.compile(
    loss=loss,
    metrics=["accuracy"],
    optimizer=keras.optimizers.Adam(1e-4),
)

history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    print(f"epoch {epoch}")
    train_cleaned = pd.read_csv(export_dir / "train_cleaned.csv")
    train_cleaned = train_cleaned.sample(frac=1)
    val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
    val_cleaned = val_cleaned.sample(frac=1)

    (train_ds, num_train_examples) = make_dataset(train_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False, augment=True)
    (val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)

    STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
    VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))
    
    epoch_hist = baseline_model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_data=val_ds,
        validation_steps=VAL_STEPS_PER_EPOCH,
        epochs=epoch+1,
        initial_epoch=epoch,
    )
    
    history["loss"].append(epoch_hist.history["loss"][-1])
    history["accuracy"].append(epoch_hist.history["accuracy"][-1])
    history["val_loss"].append(epoch_hist.history["val_loss"][-1])
    history["val_accuracy"].append(epoch_hist.history["val_accuracy"][-1])



#### eval on test set and get per class metrics

In [ ]:
test_preds_baseline_aug = baseline_model.predict(test_ds)
yhats = test_preds_baseline_aug.argmax(axis=1)
ys = test_cleaned["leaf_class_id"].values
results = pd.DataFrame({
    "y": ys,
    "yhat": yhats,
})
(results.y == results.yhat).sum() / len(results)

In [ ]:
subsets = []
for leaf_class_id in range(num_classes):
    num_train_classes = len(train_cleaned[train_cleaned.leaf_class_id==leaf_class_id])
    subset = results[(results.y == leaf_class_id)]
    subset_acc = (subset.y == subset.yhat).sum() / len(subset)
    subsets.append( (num_train_classes, subset_acc) )

(train_classes, accs) = list(zip(*subsets))
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(train_classes, accs)
b_baseline_aug, a_baseline_aug = np.polyfit(train_classes, accs, deg=1)
xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_baseline_aug + b_baseline_aug * xseq, color="k", lw=2.5)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_500 + b_500 * xseq, lw=2.5, label="500")
ax.plot(xseq, a_1k + b_1k * xseq, lw=2.5, label="1k")
ax.plot(xseq, baseline_a + baseline_b * xseq, lw=2.5, label="baseline")
ax.plot(xseq, a_baseline_aug + b_baseline_aug * xseq, lw=2.5, label="baseline with aug")

ax.set_ylim([0, 1])

ax.legend()
plt.show()

# augment with 1k samples

In [ ]:
train_plus_aug_df = pd.concat([train_cleaned, train_augmented])
max_sample_size = 1_000

model = make_net(num_classes)
model.compile(
    loss=loss,
    metrics=["accuracy"],
    optimizer=keras.optimizers.Adam(1e-4),
)

history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

NUM_EPOCHS = 10
for epoch in range(NUM_EPOCHS):
    print(f"epoch {epoch}")
    
    train_plus_aug_df = train_plus_aug_df.sample(frac=1)
    train_df_sample = train_plus_aug_df.groupby("leaf_class_id").head(max_sample_size)
    train_df_sample = train_df_sample.sample(frac=1)
    
    val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
    val_cleaned = val_cleaned.sample(frac=1)
    
    (train_ds, num_train_examples) = make_dataset(train_df_sample, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False, augment=True)
    (val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)

    STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
    VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))
    
    epoch_hist = model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_data=val_ds,
        validation_steps=VAL_STEPS_PER_EPOCH,
        epochs=epoch+1,
        initial_epoch=epoch,
    )
    history["loss"].append(epoch_hist.history["loss"][-1])
    history["accuracy"].append(epoch_hist.history["accuracy"][-1])
    history["val_loss"].append(epoch_hist.history["val_loss"][-1])
    history["val_accuracy"].append(epoch_hist.history["val_accuracy"][-1])


In [ ]:
# eval on test set and get per class samples

test_preds_1k_aug = model.predict(test_ds)
yhats = test_preds_1k_aug.argmax(axis=1)
ys = test_cleaned["leaf_class_id"].values
results = pd.DataFrame({
    "y": ys,
    "yhat": yhats,
})
(results.y == results.yhat).sum() / len(results)

In [ ]:
subsets = []
for leaf_class_id in range(num_classes):
    num_train_classes = len(train_cleaned[train_cleaned.leaf_class_id==leaf_class_id])
    subset = results[(results.y == leaf_class_id)]
    subset_acc = (subset.y == subset.yhat).sum() / len(subset)
    subsets.append( (num_train_classes, subset_acc) )

(train_classes, accs) = list(zip(*subsets))
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(train_classes, accs)
b_1k_aug, a_1k_aug = np.polyfit(train_classes, accs, deg=1)
xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_1k_aug + b_1k_aug * xseq, color="k", lw=2.5)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_500 + b_500 * xseq, lw=2.5, label="500")
ax.plot(xseq, a_1k + b_1k * xseq, lw=2.5, label="1k")
ax.plot(xseq, baseline_a + baseline_b * xseq, lw=2.5, label="baseline")
ax.plot(xseq, a_baseline_aug + b_baseline_aug * xseq, lw=2.5, label="baseline with aug")
ax.plot(xseq, a_1k_aug + b_1k_aug * xseq, lw=2.5, label="1k with aug")

ax.set_ylim([0, 1])

ax.legend()
plt.show()

# cos decay?

In [ ]:
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    decay_steps=900,
    warmup_steps=100,
    warmup_target=1e-4,
)


In [ ]:
lrs = []

for i in range(1_000):
    lrs.append(lr_schedule(step=i))

plt.plot(lrs)

In [ ]:
train_plus_aug_df = pd.concat([train_cleaned, train_augmented])
max_sample_size = 1_000
NUM_EPOCHS = 10

total_steps = STEPS_PER_EPOCH * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)
decay_steps = total_steps - warmup_steps

decay_steps = STEPS_PER_EPOCH * NUM_EPOCHS
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    decay_steps=decay_steps,
    warmup_steps=warmup_steps,
    warmup_target=1e-4,
)

model = make_net(num_classes)
model.compile(
    loss=loss,
    metrics=["accuracy"],
    optimizer=keras.optimizers.Adam(lr_schedule),
)

history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

for epoch in range(NUM_EPOCHS):
    print(f"epoch {epoch}")
    
    train_plus_aug_df = train_plus_aug_df.sample(frac=1)
    train_df_sample = train_plus_aug_df.groupby("leaf_class_id").head(max_sample_size)
    train_df_sample = train_df_sample.sample(frac=1)
    
    val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
    val_cleaned = val_cleaned.sample(frac=1)
    
    (train_ds, num_train_examples) = make_dataset(train_df_sample, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False, augment=True)
    (val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)

    STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
    VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))
    
    epoch_hist = model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_data=val_ds,
        validation_steps=VAL_STEPS_PER_EPOCH,
        epochs=epoch+1,
        initial_epoch=epoch,
    )
    history["loss"].append(epoch_hist.history["loss"][-1])
    history["accuracy"].append(epoch_hist.history["accuracy"][-1])
    history["val_loss"].append(epoch_hist.history["val_loss"][-1])
    history["val_accuracy"].append(epoch_hist.history["val_accuracy"][-1])


In [ ]:
# eval on test set and get per class samples

test_preds_1k_aug_cos = model.predict(test_ds)
yhats = test_preds_1k_aug_cos.argmax(axis=1)
ys = test_cleaned["leaf_class_id"].values
results = pd.DataFrame({
    "y": ys,
    "yhat": yhats,
})
(results.y == results.yhat).sum() / len(results)

In [ ]:
subsets = []
for leaf_class_id in range(num_classes):
    num_train_classes = len(train_cleaned[train_cleaned.leaf_class_id==leaf_class_id])
    subset = results[(results.y == leaf_class_id)]
    subset_acc = (subset.y == subset.yhat).sum() / len(subset)
    subsets.append( (num_train_classes, subset_acc) )

(train_classes, accs) = list(zip(*subsets))
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(train_classes, accs)
b_1k_aug_cos, a_1k_aug_cos = np.polyfit(train_classes, accs, deg=1)
xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_1k_aug_cos + b_1k_aug_cos * xseq, color="k", lw=2.5)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

xseq = np.linspace(0, 1_000, num=100)
ax.plot(xseq, a_500 + b_500 * xseq, lw=2.5, label="500")
ax.plot(xseq, a_1k + b_1k * xseq, lw=2.5, label="1k")
ax.plot(xseq, baseline_a + baseline_b * xseq, lw=2.5, label="baseline")
ax.plot(xseq, a_baseline_aug + b_baseline_aug * xseq, lw=2.5, label="baseline with aug")
ax.plot(xseq, a_1k_aug_cos + b_1k_aug_cos * xseq, lw=2.5, label="1k with aug cos")

ax.set_ylim([0, 1])

ax.legend()
plt.show()

# ok let's try a more production like run, more epochs and with wandb integration

In [ ]:
import wandb
from wandb.integration.keras import WandbMetricsLogger

In [ ]:
wandb.init(
    project="galliformes-new"
)

train_plus_aug_df = pd.concat([train_cleaned, train_augmented])
max_sample_size = 1_000
NUM_EPOCHS = 10

total_steps = STEPS_PER_EPOCH * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)
decay_steps = total_steps - warmup_steps

decay_steps = STEPS_PER_EPOCH * NUM_EPOCHS
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    decay_steps=decay_steps,
    warmup_steps=warmup_steps,
    warmup_target=6e-4,
)
optimizer = keras.optimizers.RMSprop(
    learning_rate=lr_schedule,
    rho=0.9,
    momentum=0.9,
    epsilon=1.0e-3,
)
loss = keras.losses.CategoricalCrossentropy(from_logits=True)

model = make_net(num_classes)
model.compile(
    loss=loss,
    metrics=["accuracy"],
    optimizer=keras.optimizers.Adam(lr_schedule),
)

history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

for epoch in range(NUM_EPOCHS):
    print(f"epoch {epoch}")
    
    train_plus_aug_df = train_plus_aug_df.sample(frac=1)
    train_df_sample = train_plus_aug_df.groupby("leaf_class_id").head(max_sample_size)
    train_df_sample = train_df_sample.sample(frac=1)
    
    val_cleaned = pd.read_csv(export_dir / "val_cleaned.csv")
    val_cleaned = val_cleaned.sample(frac=1)
    
    (train_ds, num_train_examples) = make_dataset(train_df_sample, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False, augment=True)
    (val_ds, num_val_examples) = make_dataset(val_cleaned, BATCH_SIZE, IMAGE_SIZE, repeat_forever=False)

    STEPS_PER_EPOCH = int(np.ceil(num_train_examples / BATCH_SIZE))
    VAL_STEPS_PER_EPOCH = int(np.ceil(num_val_examples / BATCH_SIZE))
    
    epoch_hist = model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_data=val_ds,
        validation_steps=VAL_STEPS_PER_EPOCH,
        epochs=epoch+1,
        initial_epoch=epoch,
        callbacks=[ WandbMetricsLogger(log_freq=100) ],
    )
    history["loss"].append(epoch_hist.history["loss"][-1])
    history["accuracy"].append(epoch_hist.history["accuracy"][-1])
    history["val_loss"].append(epoch_hist.history["val_loss"][-1])
    history["val_accuracy"].append(epoch_hist.history["val_accuracy"][-1])

wandb.finish()

In [ ]:
wandb.finish()

In [ ]:
model.save("/home/alex/export_eval_models/model_exports/INatVision_MobileNets_Galliformes.keras")

In [ ]:
!ls ~/export_eval_models/model_exports/